Sure! Here's a quick outline:

Introduction to Medical Imaging & AI: Overview of CT, PET, MRI, X-ray, and what AI can solve.
Imaging Types & Processing: Pixel values, normalization, and preprocessing tricks.
Radiomics Tips & Tricks: Extracting meaningful features from images.
Segmentations: AI-based segmentation methods and applications.
AI Model Types: Classification models and radiomics-based classification.
Survival Analysis: Using imaging and AI for prognostic modeling.
Causal & Counterfactual Analysis: Advanced AI insights.
Foundation Models in Imaging: Emerging large AI models.
Multimodal Imaging: Integrating multiple imaging types with AI.

In [ ]:
# covering dicom 2 nifti, orientation

# MRI

MRI preprocessing for foundation models

Identify the MRI sequence: T1, T2, FLAIR, DWI, ADC, post-contrast, etc.

Convert images to a consistent orientation, such as RAS.

Optionally apply N4 bias-field correction.

Resample to a common voxel spacing, often around 1×1×1 mm³.

Optionally register brain MRI to an anatomical atlas.

Optionally perform skull stripping for brain-specific models.

Crop empty background around the anatomy.

Normalize intensities using foreground z-score or percentile scaling.

Train using fixed-size 3D patches such as 96×96×96.

Apply augmentation such as rotation, scaling, noise, contrast changes, bias fields, motion, and Gibbs artifacts.

Main challenge: MRI intensity is not standardized and changes with scanner, protocol, and sequence.

Key difference
X-ray: mainly 2D resizing and intensity normalization.
MRI: requires more spatial standardization, sequence handling, voxel resampling, and 3D patch processing.

In [ ]:
# X-ray
import numpy as np
import torch
import torch.nn.functional as F
import pydicom
import nibabel as nib


# X-ray:
# DICOM → inversion check → percentile clipping → 0–1 normalization → resize
def preprocess_xray(dicom_path, output_size=512):
    # Read DICOM
    ds = pydicom.dcmread(dicom_path)
    image = ds.pixel_array.astype(np.float32)

    # Invert MONOCHROME1 images
    if ds.PhotometricInterpretation == "MONOCHROME1":
        image = image.max() - image

    # Remove extreme values
    low, high = np.percentile(image, [1, 99])
    image = np.clip(image, low, high)

    # Normalize to 0–1
    image = (image - low) / (high - low + 1e-8)

    # Convert to tensor: [batch, channel, height, width]
    image = torch.tensor(image).unsqueeze(0).unsqueeze(0)

    # Resize
    image = F.interpolate(
        image,
        size=(output_size, output_size),
        mode="bilinear",
        align_corners=False
    )

    return image.squeeze(0)  # [1, 512, 512]

xray = preprocess_xray("chest_xray.dcm")

print(xray.shape)
print(xray.min(), xray.max())

X-ray
When downloaded as JPEG/PNG

Usually already completed:

DICOM decoding
MONOCHROME1/MONOCHROME2 inversion
Window/LUT rendering
Conversion to visible grayscale pixels
Sometimes resizing

For example, MIMIC-CXR-JPG is a processed JPEG version derived from the original DICOM images. Therefore, you normally do not perform DICOM inversion again.

You still generally perform:
JPEG/PNG
→ convert to float
→ scale to [0,1]
→ resize/crop
→ model-specific mean/std normalization


Always inspect several images visually because some smaller or unofficial datasets may have inconsistent processing.

When downloaded as DICOM

You should still check:

MONOCHROME1 versus MONOCHROME2
VOI LUT or windowing
Rescale information
Image view: AP, PA, lateral
Incorrect or blank images

In [ ]:
For X-ray foundation models

Most commonly:

Scale pixels to [0,1]
Apply dataset or pretrained-model mean and standard deviation

In [ ]:
image = image.astype("float32") / 255.0
image = (image - mean) / std   #image = (image - model_mean) / model_std
#Do not independently z-score every image unless the original FM used that preprocessing.

In [ ]:
# MRI:
# NIfTI → RAS orientation → percentile clipping → foreground z-score → 3D resize
def preprocess_mri(nifti_path, output_size=(128, 128, 128)):
    # Read NIfTI
    nii = nib.load(nifti_path)

    # Convert to standard RAS orientation
    nii = nib.as_closest_canonical(nii)

    image = nii.get_fdata().astype(np.float32)

    # Define foreground
    foreground = image[image != 0]

    # Remove extreme values
    low, high = np.percentile(foreground, [1, 99])
    image = np.clip(image, low, high)

    # Z-score normalization using foreground
    foreground = image[image != 0]
    mean = foreground.mean()
    std = foreground.std()

    image = (image - mean) / (std + 1e-8)

    # Keep background equal to zero
    image[image == (-mean / (std + 1e-8))] = 0

    # Convert to tensor: [batch, channel, depth, height, width]
    image = torch.tensor(image).unsqueeze(0).unsqueeze(0)

    # Resize volume
    image = F.interpolate(
        image,
        size=output_size,
        mode="trilinear",
        align_corners=False
    )

    return image.squeeze(0)  # [1, 128, 128, 128]

mri = preprocess_mri("brain_mri.nii.gz")

print(mri.shape)
print(mri.mean(), mri.std())

In [ ]:
MRI

MRI datasets vary more.

Curated challenge datasets

Datasets such as BraTS are already heavily preprocessed:

Sequences co-registered
Standard orientation
Resampled to approximately 1×1×1 mm³
Skull stripped
Sometimes bias-field corrected

Therefore, you normally should not repeat registration or skull stripping unless the dataset documentation says otherwise.

Clinical or raw MRI datasets

A clinical NIfTI may only have been converted from DICOM. You may still need:

Orientation correction
Resampling
Bias-field correction
Registration
Foreground cropping
Intensity normalization

Also, MRI does not have X-ray-style MONOCHROME1 inversion concerns after conversion to NIfTI. However, intensity meaning still differs between T1, T2, FLAIR, and other sequences.

In [2]:
# For MRI foundation models

# Two common approaches are:

# MRI-specific model: foreground z-score

mask = image != 0
image[mask] = (image[mask] - image[mask].mean()) / (image[mask].std() + 1e-8)
#This is very common for MRI because MRI has no standardized physical intensity scale.

NameError: name 'image' is not defined

In [ ]:
# Large heterogeneous foundation model: percentile scaling
# Large foundation models often use percentile scaling because it works consistently across different organs, sequences, and datasets. 
# For example, 3DINO mapped the 0.05th and 99.95th percentiles to −1 and 1.
mask = image != 0
low, high = np.percentile(image[mask], [0.5, 99.5])

image = np.clip(image, low, high)
image = (image - low) / (high - low + 1e-8)

In [ ]:
Best practical rule
Data	Recommended default
X-ray JPEG/PNG	[0,1], followed by the FM’s required mean/std
X-ray DICOM	Render/invert first, then [0,1] and model normalization
MRI-specific model	Foreground z-score
Large heterogeneous MRI FM	Percentile clipping and scaling to [0,1] or [-1,1]
Existing pretrained FM	Exactly reproduce its original preprocessing

When using a pretrained foundation model, use the normalization used during its pretraining.